In [0]:
%pip install python-docx openpyxl python-pptx
dbutils.library.restartPython()

In [0]:
import os

COMPANY_NAME = dbutils.secrets.get("uc13", "sp_company_name")
#VOLUME_PATH = f"/Volumes/uc13/ingestion/raw_files/{COMPANY_NAME}"
VOLUME_PATH = f"/Volumes/uc13/ingestion/raw_files"
CATALOG = "uc13"
SCHEMA = "ingestion"
TABLE_CHUNKS = f"{CATALOG}.{SCHEMA}.chunks"
TABLE_EMBEDDINGS = f"{CATALOG}.{SCHEMA}.embeddings"
EMBEDDING_ENDPOINT = "databricks-bge-large-en"

print(f"Company: {COMPANY_NAME}")
print(f"Volume path: {VOLUME_PATH}")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS uc13.ingestion.chunks (
  chunk_id       STRING,
  doc_id         STRING,
  file_name      STRING,
  file_type      STRING,
  relative_path  STRING,
  chunk_index    INT,
  chunk_text     STRING,
  section_header STRING,
  page_start     INT,
  page_end       INT,
  tab            STRING,
  char_count     INT,
  created_at     TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS uc13.ingestion.embeddings (
  chunk_id   STRING,
  doc_id     STRING,
  file_name  STRING,
  embedding  ARRAY<FLOAT>,
  created_at TIMESTAMP
) USING DELTA;

In [0]:
import json
import uuid
import hashlib
import os
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    file_name: str
    file_type: str
    relative_path: str
    chunk_index: int
    chunk_text: str
    section_header: Optional[str] = None
    page_start: Optional[int] = None
    page_end: Optional[int] = None
    tab: Optional[str] = None
    char_count: int = 0

def make_doc_id(path):
    return hashlib.md5(path.encode()).hexdigest()

SKIP_TYPES = {"page_footer", "page_number", "header"}
HEADER_TYPES = {"section_header", "title"}

def parse_pdf_with_ai(file_path, doc_id, file_name):
    """Uses ai_parse_document to extract structured chunks from PDF."""
    try:
        row = spark.sql(f"""
            SELECT to_json(ai_parse_document(
                content,
                map('version', '2.0')
            )) as parsed
            FROM read_files(
                '{file_path}',
                format => 'binaryFile'
            )
        """).collect()[0]["parsed"]

        result = json.loads(row)

        elements = result["document"]["elements"]
        chunks = []
        current_header = None
        current_texts = []
        current_pages = []
        chunk_index = 0

        def flush_chunk():
            nonlocal chunk_index
            if not current_texts:
                return
            text = "\n".join(current_texts).strip()
            if len(text) < 50:
                return
            chunk_text = f"{current_header}\n\n{text}" if current_header else text
            chunks.append(Chunk(
                chunk_id=str(uuid.uuid4()),
                doc_id=doc_id,
                file_name=file_name,
                file_type="pdf",
                relative_path=file_path,
                chunk_index=chunk_index,
                chunk_text=chunk_text,
                section_header=current_header,
                page_start=min(current_pages) + 1 if current_pages else None,
                page_end=max(current_pages) + 1 if current_pages else None,
                char_count=len(chunk_text)
            ))
            chunk_index += 1

        for el in elements:
            el_type = el.get("type", "")
            content = el.get("content", "").strip()
            page_id = el.get("page_id")

            if el_type in SKIP_TYPES or not content:
                continue

            if el_type in HEADER_TYPES:
                flush_chunk()
                current_header = content
                current_texts = []
                current_pages = []
            else:
                current_texts.append(content)
                if page_id is not None:
                    current_pages.append(page_id)

        flush_chunk()  # flush last section

        print(f"  ✓ {file_name}: {len(chunks)} chunks")
        return chunks

    except Exception as e:
        print(f"  ✗ {file_name}: {e}")
        return []

print("PDF parser ready")

In [0]:
def parse_excel(file_path, doc_id, file_name):
    import openpyxl
    chunks = []
    ROWS_PER_CHUNK = 100
    try:
        wb = openpyxl.load_workbook(file_path, read_only=True, data_only=True)
        chunk_index = 0
        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            all_rows = []
            headers = None
            for row in ws.iter_rows(values_only=True):
                vals = [str(c) if c is not None else "" for c in row]
                if not any(v.strip() for v in vals):
                    continue
                if headers is None:
                    headers = vals
                    continue
                all_rows.append(vals)

            if not headers:
                continue

            header_str = "Headers: " + " | ".join(headers)

            if len(all_rows) <= ROWS_PER_CHUNK:
                # Sheet fits in one chunk
                rows_text = [header_str]
                for row in all_rows:
                    row_str = " | ".join(
                        f"{headers[i]}: {row[i]}"
                        for i in range(min(len(headers), len(row)))
                        if row[i].strip()
                    )
                    if row_str:
                        rows_text.append(row_str)
                chunk_text = f"Sheet: {sheet_name}\n\n" + "\n".join(rows_text)
                chunks.append(Chunk(
                    chunk_id=str(uuid.uuid4()),
                    doc_id=doc_id,
                    file_name=file_name,
                    file_type="xlsx",
                    relative_path=file_path,
                    chunk_index=chunk_index,
                    chunk_text=chunk_text,
                    section_header=f"Sheet: {sheet_name}",
                    tab=sheet_name,
                    char_count=len(chunk_text)
                ))
                chunk_index += 1
            else:
                # Split into batches of ROWS_PER_CHUNK
                for batch_start in range(0, len(all_rows), ROWS_PER_CHUNK):
                    batch = all_rows[batch_start:batch_start + ROWS_PER_CHUNK]
                    batch_end = batch_start + len(batch)
                    rows_text = [header_str]
                    for row in batch:
                        row_str = " | ".join(
                            f"{headers[i]}: {row[i]}"
                            for i in range(min(len(headers), len(row)))
                            if row[i].strip()
                        )
                        if row_str:
                            rows_text.append(row_str)
                    chunk_text = (
                        f"Sheet: {sheet_name} "
                        f"(rows {batch_start + 1}–{batch_end})\n\n"
                        + "\n".join(rows_text)
                    )
                    chunks.append(Chunk(
                        chunk_id=str(uuid.uuid4()),
                        doc_id=doc_id,
                        file_name=file_name,
                        file_type="xlsx",
                        relative_path=file_path,
                        chunk_index=chunk_index,
                        chunk_text=chunk_text,
                        section_header=f"Sheet: {sheet_name} rows {batch_start + 1}–{batch_end}",
                        tab=sheet_name,
                        char_count=len(chunk_text)
                    ))
                    chunk_index += 1

        print(f"  ✓ {file_name}: {len(chunks)} chunks (sheets/batches)")
    except Exception as e:
        print(f"  ✗ {file_name}: {e}")
    return chunks
    
print("Excel parser ready")

In [0]:
def parse_word(file_path, doc_id, file_name):
    from docx import Document
    chunks = []
    try:
        doc = Document(file_path)
        current_header = None
        current_texts = []
        chunk_index = 0

        def flush_chunk():
            nonlocal chunk_index
            if not current_texts:
                return
            text = "\n".join(current_texts).strip()
            if len(text) < 50:
                return
            chunk_text = f"{current_header}\n\n{text}" if current_header else text
            chunks.append(Chunk(
                chunk_id=str(uuid.uuid4()),
                doc_id=doc_id,
                file_name=file_name,
                file_type="docx",
                relative_path=file_path,
                chunk_index=chunk_index,
                chunk_text=chunk_text,
                section_header=current_header,
                char_count=len(chunk_text)
            ))
            chunk_index += 1

        for para in doc.paragraphs:
            if not para.text.strip():
                continue
            if para.style.name.startswith("Heading"):
                flush_chunk()
                current_header = para.text.strip()
                current_texts = []
            else:
                current_texts.append(para.text.strip())

        flush_chunk()
        print(f"  ✓ {file_name}: {len(chunks)} chunks (sections)")
    except Exception as e:
        print(f"  ✗ {file_name}: {e}")
    return chunks

print("Word parser ready")

In [0]:
def parse_csv(file_path, doc_id, file_name):
    import csv
    chunks = []
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            reader = csv.reader(f)
            rows = list(reader)

        if not rows:
            return chunks

        headers = rows[0]
        BATCH_SIZE = 50
        chunk_index = 0

        for start in range(1, len(rows), BATCH_SIZE):
            batch = rows[start:start + BATCH_SIZE]
            lines = ["Headers: " + " | ".join(headers)]
            for row in batch:
                row_str = " | ".join(
                    f"{headers[i]}: {row[i]}"
                    for i in range(min(len(headers), len(row)))
                    if row[i].strip()
                )
                if row_str:
                    lines.append(row_str)

            chunk_text = "\n".join(lines)
            chunks.append(Chunk(
                chunk_id=str(uuid.uuid4()),
                doc_id=doc_id,
                file_name=file_name,
                file_type="csv",
                relative_path=file_path,
                chunk_index=chunk_index,
                chunk_text=chunk_text,
                section_header=f"Rows {start}-{start+len(batch)-1}",
                char_count=len(chunk_text)
            ))
            chunk_index += 1

        print(f"  ✓ {file_name}: {len(chunks)} chunks ({len(rows)-1} rows)")
    except Exception as e:
        print(f"  ✗ {file_name}: {e}")
    return chunks

print("CSV parser ready")

In [0]:
ALLOWED = {".pdf", ".xlsx", ".xls", ".docx", ".doc", ".csv"}

def parse_file(file_path):
    file_name = os.path.basename(file_path)
    ext = os.path.splitext(file_name)[1].lower()
    doc_id = make_doc_id(file_path)
    if ext == ".pdf":
        return parse_pdf_with_ai(file_path, doc_id, file_name)
    elif ext in {".xlsx", ".xls"}:
        return parse_excel(file_path, doc_id, file_name)
    elif ext in {".docx", ".doc"}:
        return parse_word(file_path, doc_id, file_name)
    elif ext == ".csv":
        return parse_csv(file_path, doc_id, file_name)
    else:
        print(f"  — skipped: {file_name}")
        return []

# Encontrar todos los archivos
all_files = []
for root, dirs, files in os.walk(VOLUME_PATH):
    for f in files:
        if not f.startswith(".") and os.path.splitext(f)[1].lower() in ALLOWED:
            all_files.append(os.path.join(root, f))

print(f"Found {len(all_files)} files\n")

# Parsear todos
all_chunks = []
for file_path in all_files:
    chunks = parse_file(file_path)
    all_chunks.extend(chunks)

print(f"\n--- Summary ---")
print(f"Total chunks: {len(all_chunks)}")
print(f"Total characters: {sum(c.char_count for c in all_chunks):,}")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)
from datetime import datetime

schema = StructType([
    StructField("chunk_id",       StringType(),    False),
    StructField("doc_id",         StringType(),    False),
    StructField("file_name",      StringType(),    False),
    StructField("file_type",      StringType(),    False),
    StructField("relative_path",  StringType(),    False),
    StructField("chunk_index",    IntegerType(),   False),
    StructField("chunk_text",     StringType(),    False),
    StructField("section_header", StringType(),    True),
    StructField("page_start",     IntegerType(),   True),
    StructField("page_end",       IntegerType(),   True),
    StructField("tab",            StringType(),    True),
    StructField("char_count",     IntegerType(),   False),
    StructField("created_at",     TimestampType(), False),
])

rows = [
    Row(
        chunk_id=c.chunk_id,
        doc_id=c.doc_id,
        file_name=c.file_name,
        file_type=c.file_type,
        relative_path=c.relative_path,
        chunk_index=int(c.chunk_index),
        chunk_text=c.chunk_text,
        section_header=c.section_header,
        page_start=int(c.page_start) if c.page_start is not None else None,
        page_end=int(c.page_end) if c.page_end is not None else None,
        tab=c.tab,
        char_count=int(c.char_count),
        created_at=datetime.now()
    )
    for c in all_chunks
]

df_chunks = spark.createDataFrame(rows, schema=schema)
df_chunks.write.mode("overwrite").saveAsTable(TABLE_CHUNKS)
print(f"Saved {df_chunks.count()} chunks to {TABLE_CHUNKS}")

In [0]:
import mlflow.deployments
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, FloatType, TimestampType
)
from datetime import datetime

client = mlflow.deployments.get_deploy_client("databricks")

def get_embeddings_batch(texts, batch_size=20):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = [t[:8000] for t in texts[i:i+batch_size]]
        response = client.predict(
            endpoint=EMBEDDING_ENDPOINT,
            inputs={"input": batch}
        )
        embeddings.extend([item["embedding"] for item in response["data"]])
        if i % 100 == 0:
            print(f"  Embedded {i}/{len(texts)} chunks...")
    return embeddings

print(f"Generating embeddings for {len(all_chunks)} chunks...")
texts = [c.chunk_text for c in all_chunks]
embeddings = get_embeddings_batch(texts)
print(f"Generated {len(embeddings)} embeddings")

schema_emb = StructType([
    StructField("chunk_id",   StringType(),              False),
    StructField("doc_id",     StringType(),              False),
    StructField("file_name",  StringType(),              False),
    StructField("embedding",  ArrayType(FloatType()),    False),
    StructField("created_at", TimestampType(),           False),
])

emb_rows = [
    Row(
        chunk_id=all_chunks[i].chunk_id,
        doc_id=all_chunks[i].doc_id,
        file_name=all_chunks[i].file_name,
        embedding=[float(x) for x in embeddings[i]],
        created_at=datetime.now()
    )
    for i in range(len(all_chunks))
]

df_emb = spark.createDataFrame(emb_rows, schema=schema_emb)
df_emb.write.mode("overwrite").saveAsTable(TABLE_EMBEDDINGS)
print(f"Saved {df_emb.count()} embeddings to {TABLE_EMBEDDINGS}")

In [0]:
%sql
SELECT 
  file_type,
  COUNT(DISTINCT doc_id) as documents,
  COUNT(*) as total_chunks,
  AVG(char_count) as avg_chars_per_chunk,
  MIN(char_count) as min_chars,
  MAX(char_count) as max_chars
FROM uc13.ingestion.chunks
GROUP BY file_type
ORDER BY file_type

In [0]:
import os
os.environ["SP_CLIENT_ID"]     = dbutils.secrets.get("uc13", "sp_client_id")
os.environ["SP_CLIENT_SECRET"] = dbutils.secrets.get("uc13", "sp_client_secret")
os.environ["SP_TENANT_ID"]     = dbutils.secrets.get("uc13", "sp_tenant_id")
os.environ["SP_SITE_URL"]      = dbutils.secrets.get("uc13", "sp_site_url")
os.environ["SP_FOLDER_PATH"]   = dbutils.secrets.get("uc13", "sp_folder_path")
os.environ["SP_COMPANY_NAME"]  = dbutils.secrets.get("uc13", "sp_company_name")
os.environ["UC_VOLUME_PATH"]   = "/Volumes/uc13/ingestion/raw_files"

import sys
sys.path.insert(0, "/Workspace/Users/hector.corro@nimblegravity.com/Rallyday/databricks")

from agents.ingestion.tools.connector import list_files
from agents.ingestion.tools.uploader import upload_batch, FilePayload
from agents.ingestion.tools.connector import download_file

# Lista todos los archivos deduplicados
files = list_files()
print(f"Files to download and upload: {len(files)}")

In [0]:
import os
print(f"Company: {os.environ.get('SP_COMPANY_NAME')}")
print(f"Destination volume: /Volumes/uc13/ingestion/raw_files/{os.environ.get('SP_COMPANY_NAME')}")

# Verifica que el Volume existe y es escribible
test_path = f"/Volumes/uc13/ingestion/raw_files/{os.environ.get('SP_COMPANY_NAME')}"
os.makedirs(test_path, exist_ok=True)
print(f"Volume path accessible: {os.path.exists(test_path)}")

### Download all batch size into volume

In [0]:
from agents.ingestion.tools.uploader import upload_batch, FilePayload
from agents.ingestion.tools.connector import download_file, list_files, authenticate, get_site_id
from concurrent.futures import ThreadPoolExecutor
import os
import requests

DEST_VOLUME = f"/Volumes/uc13/ingestion/raw_files/{os.environ['SP_COMPANY_NAME']}"

def download_and_upload(file_meta):
    try:
        # Descarga bytes directamente sin guardar a disco
        token = authenticate()
        site_id = get_site_id(token)
        url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/items/{file_meta.item_id}/content"
        response = requests.get(url, headers={"Authorization": f"Bearer {token}"}, allow_redirects=True)
        response.raise_for_status()
        content = response.content
        
        payload = FilePayload(
            file_name=file_meta.name,
            relative_path=file_meta.relative_path,
            content=content,
            size_bytes=len(content)
        )
        return payload
    except Exception as e:
        print(f"  ✗ {file_meta.name}: {e}")
        return None

BATCH_SIZE = 50
total_success = 0
total_failed = 0
num_batches = -(-len(files) // BATCH_SIZE)

for batch_start in range(0, len(files), BATCH_SIZE):
    batch = files[batch_start:batch_start + BATCH_SIZE]
    print(f"\nBatch {batch_start//BATCH_SIZE + 1}/{num_batches}: files {batch_start+1}-{min(batch_start+BATCH_SIZE, len(files))}")
    
    with ThreadPoolExecutor(max_workers=5) as executor:
        payloads = list(executor.map(download_and_upload, batch))
    
    valid_payloads = [p for p in payloads if p is not None]
    
    if valid_payloads:
        summary = upload_batch(valid_payloads)
        total_success += summary.successful
        total_failed += summary.failed + (len(batch) - len(valid_payloads))
    
    print(f"  Batch done: {len(valid_payloads)} uploaded, {len(batch)-len(valid_payloads)} failed")

print(f"\n=== Final summary ===")
print(f"Total successful: {total_success}")
print(f"Total failed:     {total_failed}")
print(f"Destination:      {DEST_VOLUME}")